In [2]:
import numpy as np
import pandas as pd
import pyedflib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Flatten, Dense

In [3]:
data = pd.read_csv('to_train_data.csv')

In [4]:
data = data[['FrL', 'FrR', 'OcR', 'target', 'FrL_RSI_100', 'FrR_RSI_100', 'OcR_RSI_100', 'FrL_DPO_100', 'FrR_DPO_100', 'OcR_DPO_100', 'FrL_momentum_100', 'FrR_momentum_100', 'OcR_momentum_100', 'FrL_stachostic_100', 'FrR_stachostic_100', 'OcR_stachostic_100']]

In [5]:
data.shape

(156000, 16)

In [6]:
data

,FrL,FrR,OcR,target,FrL_RSI_100,FrR_RSI_100,OcR_RSI_100,FrL_DPO_100,FrR_DPO_100,OcR_DPO_100,FrL_momentum_100,FrR_momentum_100,OcR_momentum_100,FrL_stachostic_100,FrR_stachostic_100,OcR_stachostic_100
0,-0.167625,-0.125625,-0.089688,0,46.093039,44.694688,40.596906,-0.241563,-0.332437,-0.388313,-0.241563,-0.332437,-0.388313,53.231461,64.346005,41.506863
1,-0.175687,-0.100250,-0.038562,0,45.279088,44.649417,42.444670,-0.288938,-0.335000,-0.316625,-0.288938,-0.335000,-0.316625,52.378117,66.847391,49.469483
2,-0.176687,-0.123250,-0.039438,0,44.778364,44.303898,42.211582,-0.316875,-0.359125,-0.325313,-0.316875,-0.359125,-0.325313,52.272276,64.580124,50.842697
3,-0.174375,-0.127812,-0.014687,0,44.553370,44.236557,43.278816,-0.329187,-0.363875,-0.281500,-0.329187,-0.363875,-0.281500,52.517034,65.129521,55.635882
4,-0.138375,-0.085625,0.032250,0,45.006051,45.191817,44.714160,-0.304312,-0.306125,-0.225375,-0.304312,-0.306125,-0.225375,56.327314,69.811677,63.632641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155995,-0.029687,-0.030187,0.021687,1,49.815806,50.262162,50.460799,-0.006625,0.017500,0.013187,-0.006625,0.017500,0.013187,52.787140,36.766275,61.983823
155996,-0.035187,-0.028313,0.023062,1,50.736231,50.776644,51.625708,0.026000,0.051375,0.045562,0.026000,0.051375,0.045562,50.505574,37.299893,62.920392
155997,-0.060187,-0.015437,-0.001375,1,50.134658,50.459291,50.778680,0.004812,0.030187,0.022187,0.004812,0.030187,0.022187,40.134820,40.964070,46.275011
155998,-0.048000,-0.000625,-0.023062,1,49.632587,49.205390,49.778569,-0.013000,-0.050937,-0.006375,-0.013000,-0.050937,-0.006375,45.190563,45.179651,31.502767


In [7]:
 window_size = 200

num_samples = (len(data) // window_size) * window_size
data = data.iloc[:num_samples]  # Обрезаем лишние строки

X = data.drop(columns=['target']).values.reshape(-num_samples, window_size, 15)  # Преобразуем в форму (Batch Size, 800, 3)
y = data['target'].values[:num_samples:window_size]  # Каждое значение 'target' соответствует одному окну из 800

# Проверим итоговые формы X и y
print("Форма X:", X.shape)  # Должно быть (Количество выборок, 800, 3)
print("Форма y:", y.shape)  # Должно быть (Количество выборок,)

# Разделение на обучающую и валидационную выборки
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Определение модели
model = Sequential([
    Input(shape=(200, 15)),  # Входной слой с формой (800, 3)
    Bidirectional(LSTM(200, return_sequences=True)),
    Dropout(0.4),
    Flatten(),
    Dense(3, activation='softmax')  # Изменяем на 3, поскольку у нас 3 класса (ds, is, swd)
])

# Компиляция модели
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Вывод структуры модели
model.summary()

# Обучение модели
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# Оценка модели
model.evaluate(X_val, y_val)

Форма X: (780, 200, 15)
Форма y: (780,)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)        │ (None, 200, 400)            │         345,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 200, 400)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 80000)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 3)                   │         240,003 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 585,603 (2.23 MB)

 Trainable params: 585,603 (2.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 160ms/step - accuracy: 0.4932 - loss: 3.2066 - val_accuracy: 0.7756 - val_loss: 0.5142
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 140ms/step - accuracy: 0.8124 - loss: 0.5241 - val_accuracy: 0.8013 - val_loss: 0.5264
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 182ms/step - accuracy: 0.8837 - loss: 0.3203 - val_accuracy: 0.8141 - val_loss: 0.4661
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 210ms/step - accuracy: 0.8961 - loss: 0.2208 - val_accuracy: 0.7244 - val_loss: 0.7906
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 216ms/step - accuracy: 0.9293 - loss: 0.1754 - val_accuracy: 0.7436 - val_loss: 0.6965
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 210ms/step - accuracy: 0.9379 - loss: 0.1532 - val_accuracy: 0.7436 - val_loss: 0.6105
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 211ms/step - accuracy: 0.9725 - loss: 0.0735 - val_accuracy: 0.7821 - val_loss: 0.5704
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 212ms/step - accuracy: 0.9998 - loss: 0.0322 - val_accuracy: 0.

[0.6245771050453186, 0.7884615659713745]